# NB_hospital_daily_summary

Builds `<catalog>.gold.hospital_daily_summary` (one row per hospital per day) from silver lab_results, enriched with hospitals and calendar, with a LAG-based day-over-day test volume trend. Full rebuild per the Silver-to-Gold STTM.

In [0]:
# Parameterisation disabled: catalog is hard-coded as adb_caresync in the SQL cells below
# # Unity Catalog catalog name (STTM placeholder <catalog>)
# dbutils.widgets.text("catalog", "adb_caresync")
# print("catalog =", dbutils.widgets.get("catalog"))

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_caresync.gold;

CREATE OR REPLACE TABLE adb_caresync.gold.hospital_daily_summary AS
WITH daily AS (
  -- Driving table: lab_results grouped by hospital and test date
  SELECT
    hospital_id,
    test_date                              AS summary_date,
    COUNT(*)                               AS total_tests,
    COUNT(DISTINCT patient_id)             AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)              AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')   AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')   AS critical_test_count
  FROM adb_caresync.silver.lab_results
  GROUP BY hospital_id, test_date
),
with_trend AS (
  SELECT
    d.*,
    LAG(d.total_tests) OVER (PARTITION BY d.hospital_id ORDER BY d.summary_date) AS prev_day_total_tests
  FROM daily d
)
SELECT
  CAST(t.hospital_id AS INT)                     AS hospital_id,
  CAST(h.hospital_name AS STRING)                AS hospital_name,
  CAST(t.summary_date AS DATE)                   AS summary_date,
  CAST(c.day_name AS STRING)                     AS day_name,
  CAST(c.is_weekend AS BOOLEAN)                  AS is_weekend,
  CAST(t.total_tests AS INT)                     AS total_tests,
  CAST(t.unique_patients_tested AS INT)          AS unique_patients_tested,
  CAST(t.active_doctor_count AS INT)             AS active_doctor_count,
  CAST(t.abnormal_test_count AS INT)             AS abnormal_test_count,
  CAST(t.critical_test_count AS INT)             AS critical_test_count,
  CAST(ROUND(t.abnormal_test_count * 100.0 / t.total_tests, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(ROUND(t.critical_test_count * 100.0 / t.total_tests, 2) AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(t.prev_day_total_tests AS INT)            AS prev_day_total_tests,
  CASE
    WHEN t.prev_day_total_tests IS NULL OR t.prev_day_total_tests = 0 THEN NULL
    ELSE CAST(ROUND((t.total_tests - t.prev_day_total_tests) * 100.0 / t.prev_day_total_tests, 2) AS DECIMAL(6,2))
  END                                            AS test_volume_change_pct
FROM with_trend t
LEFT JOIN adb_caresync.silver.hospitals h
  ON t.hospital_id = h.hospital_id
LEFT JOIN adb_caresync.silver.calendar c
  ON c.date_key = CAST(date_format(t.summary_date, 'yyyyMMdd') AS INT);

In [0]:
%sql
-- Grain check: one row per (hospital_id, summary_date); totals reconcile to silver.lab_results
SELECT
  COUNT(*)                                              AS gold_rows,
  COUNT(DISTINCT hospital_id, summary_date)             AS distinct_hospital_days,
  SUM(total_tests)                                      AS sum_total_tests,
  (SELECT COUNT(*) FROM adb_caresync.silver.lab_results) AS source_lab_results,
  COUNT_IF(hospital_name IS NULL)                       AS missing_hospital_name,
  COUNT_IF(day_name IS NULL)                            AS missing_calendar,
  COUNT_IF(prev_day_total_tests IS NULL)                AS first_day_rows,
  COUNT(DISTINCT hospital_id)                           AS distinct_hospitals,
  MIN(summary_date)                                     AS min_date,
  MAX(summary_date)                                     AS max_date
FROM adb_caresync.gold.hospital_daily_summary;

In [0]:
%sql
SELECT * FROM adb_caresync.gold.hospital_daily_summary ORDER BY hospital_id, summary_date LIMIT 20;

In [0]:

# DBTITLE 1,Return hospital daily summary record count
updated_df = spark.table("adb_caresync.gold.hospital_daily_summary")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))